# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
# ==============================================================================
# Setup: Libraries, Data Ingestion, Model Training & Probability Scoring
# ==============================================================================
import duckdb
import numpy as np
import pandas as pd
from pathlib import Path
from google.colab import userdata
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

# 1. Output Directory Setup
OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Connection Setup
HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');")

FACT_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
DIM_PATH  = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

# 3. Data Query & Aggregation
raw_df = con.execute(f"""
    SELECT
        f.client_hash_id, f.content_hash_id, f.gsc_clicks, f.gsc_impressions,
        f.ga4_total_engagement_sec, c.word_count, c.backlinks,
        CASE
            WHEN (f.sessions_ai / (f.gsc_clicks + 1.0) > 0.35) AND (f.sessions_ai >= 5)
            THEN 1 ELSE 0
        END AS is_high_ai_spike
    FROM read_parquet('{FACT_PATH}') f
    JOIN read_parquet('{DIM_PATH}') c ON f.content_hash_id = c.content_hash_id
    WHERE f.gsc_data_available IS TRUE
      AND c.is_published IS TRUE
      AND c.is_deleted IS FALSE
""").df()

frame = raw_df.groupby(["client_hash_id", "content_hash_id"]).agg(
    gsc_clicks=("gsc_clicks", "sum"),
    gsc_impressions=("gsc_impressions", "sum"),
    ga4_total_engagement_sec=("ga4_total_engagement_sec", "sum"),
    word_count=("word_count", "max"),
    backlinks=("backlinks", "max"),
    is_high_ai_spike=("is_high_ai_spike", "max")
).reset_index()

# 4. Data Splitting & Feature Scaling
FEATURES = ["gsc_clicks", "gsc_impressions", "ga4_total_engagement_sec", "word_count", "backlinks"]
X = frame[FEATURES].fillna(0)
y = frame["is_high_ai_spike"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=frame['client_hash_id']))

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X.iloc[train_idx])
X_test_scaled = scaler.transform(X.iloc[test_idx])

# 5. Train Model & Score the Test Queue
model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y.iloc[train_idx])

test_results = frame.iloc[test_idx].copy()
test_results["model_score"] = model.decision_function(X_test_scaled)
print(f"Setup complete. Target queue isolated to {len(test_results):,} test pages.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Setup complete. Target queue isolated to 25,229 test pages.


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [7]:
# ==============================================================================
# 1. Ranked actions + reason codes
# ==============================================================================

# 1. Rank the queue strictly by the model's raw point score (log-odds)
df_queue = test_results.sort_values(by="model_score", ascending=False).reset_index(drop=True)

# 2. Fix the NaN issue: fill missing values in our target features with 0
features_to_clean = ["word_count", "ga4_total_engagement_sec", "backlinks"]
df_queue[features_to_clean] = df_queue[features_to_clean].fillna(0)

# 3. Define feature thresholds based on medians and 75th percentiles
wc_thresh = df_queue["word_count"].median()
eng_thresh = df_queue["ga4_total_engagement_sec"].median()
impr_thresh = df_queue["gsc_impressions"].quantile(0.75)

# 4. Construct Reason Codes using decision-support logic
reason_conditions = [
    # Top archetypes: High depth & engagement, with 0 backlinks (since model heavily penalized links)
    ((df_queue["word_count"] >= wc_thresh) & (df_queue["ga4_total_engagement_sec"] >= eng_thresh) & (df_queue["backlinks"] == 0)).to_numpy(dtype=bool),
    # Secondary archetype: High visibility/impressions but lacking depth
    ((df_queue["gsc_impressions"] >= impr_thresh) & (df_queue["word_count"] < wc_thresh)).to_numpy(dtype=bool)
]

reason_choices = [
    "HIGH_DEPTH_ENGAGEMENT_LOW_AUTHORITY",
    "HIGH_VISIBILITY_THIN_CONTENT"
]

df_queue["reason_code"] = np.select(reason_conditions, reason_choices, default="BASELINE_MONITOR")

# 5. Assign Actions to those Reason Codes
action_conditions = [
    (df_queue["reason_code"] == "HIGH_DEPTH_ENGAGEMENT_LOW_AUTHORITY").to_numpy(dtype=bool),
    (df_queue["reason_code"] == "HIGH_VISIBILITY_THIN_CONTENT").to_numpy(dtype=bool)
]

action_choices = [
    "REVIEW_FOR_AI_SNIPPET_OPTIMIZATION",
    "EXPAND_STRUCTURAL_DEPTH"
]

df_queue["action"] = np.select(action_conditions, action_choices, default="MONITOR_PERFORMANCE")

# 6. Preview the top 10 prioritized actions
print("=== Top 10 Prioritized Content Actions ===")
display_cols = ["client_hash_id", "content_hash_id", "model_score", "action", "reason_code"]

# Added floatfmt=".3f" so you can clearly see the decimal variance in the point system
print(df_queue[display_cols].head(10).to_markdown(index=False, floatfmt=".3f"))

=== Top 10 Prioritized Content Actions ===
| client_hash_id          | content_hash_id          |   model_score | action                             | reason_code                         |
|:------------------------|:-------------------------|--------------:|:-----------------------------------|:------------------------------------|
| client_9958f0a7ae1df715 | content_01ab236521ddf2f9 |        36.953 | EXPAND_STRUCTURAL_DEPTH            | HIGH_VISIBILITY_THIN_CONTENT        |
| client_3f0ce4d44fe94f3d | content_fbd410a0ce9b6c3a |        32.662 | REVIEW_FOR_AI_SNIPPET_OPTIMIZATION | HIGH_DEPTH_ENGAGEMENT_LOW_AUTHORITY |
| client_9958f0a7ae1df715 | content_bda478d54caf6a8c |        30.008 | MONITOR_PERFORMANCE                | BASELINE_MONITOR                    |
| client_e5c2aa26a8598242 | content_ef7013c86d07aa99 |        22.209 | REVIEW_FOR_AI_SNIPPET_OPTIMIZATION | HIGH_DEPTH_ENGAGEMENT_LOW_AUTHORITY |
| client_fef1a8f436438636 | content_6b4ba5a247ea6100 |        20.854 | EXPAND_STR

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 2. Intended use and limits

**Intended Use:**
This playbook is a decision-support tool designed for content strategists and human reviewers. It takes a portfolio of thousands of pages and ranks them into a prioritized queue. It highlights pages that exhibit the structural patterns (high depth, strong engagement, low traditional backlinks) typically associated with AI-referral spikes in our observed dataset. By prioritizing these specific interventions, content teams can help clients diversify their traffic sources, capturing highly engaged audiences through LLM citations even when traditional organic search visibility remains flat.

**Limits & Where it Stops Being Valid:**
* **Not Causal:** The model weights establish a directional association, not a causal guarantee. Expanding a page's word count does not guarantee an algorithm will reward it with AI traffic.
* **The Missing Data Penalty:** Approximately 33-42% of the portfolio is missing `word_count`, `ga4_total_engagement`, or `backlinks` due to pipeline delays. Because the model fills these missing values with zero, newer or uncrawled pages are inherently penalized and pushed to the bottom of the queue.
* **Context Blindness:** The model relies purely on structural numbers. It cannot distinguish between a page that is dangerously thin (and needs expansion) versus a page that is intentionally short by design (e.g., contact portals, login pages, or image galleries).

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.